# 🏰 Lakehouse Storage Layer with DuckLake & DuckDB Catalog

This notebook provides a hands-on guide to **DuckLake** as a modern Lakehouse storage layer using **DuckDB** for both compute engine and metadata catalog (`hr_lake`).

--- 

## 🎯 Objectives & Core Capabilities
- **Lakehouse Catalog (`hr_lake`) & Decoupled Storage**: Store metadata in `lakehouse/hr_lake_catalog.db` and raw data files as Parquet in `lakehouse/hr_lake_data/`.
- **Transactions (ACID Atomicity & Consistency)**: Execute multi-statement writes safely with atomic commits and rollbacks during payroll processing.
- **Time Travel**: Query past versions and historical snapshots of tables without restoring backups.
- **Schema Evolution**: Modify table structures in-place (e.g., adding a performance rating column) without rewriting existing Parquet data files.

--- 

## 📌 DuckDB Engine vs. DuckLake Storage Layer
- **DuckDB Engine**: An in-process analytical SQL query engine responsible for parsing SQL, planning execution, and processing data in memory.
- **DuckLake Storage Layer (`hr_lake`)**: An open Lakehouse storage specification and extension for DuckDB that manages metadata catalogs, ACID transactions, and decoupled Parquet data storage.

In [30]:
# DuckDB is the only runtime dependency required by this notebook.
# Install it once in the selected Python environment if needed:
# %pip install duckdb
print("Use the selected Python environment with DuckDB installed.")

Use the selected Python environment with DuckDB installed.


### DuckLake Extension & DuckDB Catalog Configuration
- **Plugin Required**: `ducklake` (installed and loaded by the connection cell).
- **Catalog**: `lakehouse/hr_lake_catalog.db` stores table schemas, snapshots, transactions, and file tracking.
- **Data Path**: `lakehouse/hr_lake_data/` stores the physical Parquet data files.
- **Execution Model**: DuckDB runs in memory for this notebook; DuckLake persists the catalog and Parquet data on disk.
- **Path Handling**: The connection cell uses absolute paths so output location does not depend on VS Code's current working directory.

In [1]:
# Step 1A: Reset the local lakehouse so this notebook starts from a known state.
# WARNING: This deletes the catalog and all Parquet data from earlier runs.
import os
import shutil
import duckdb

DATA_LAKE_DIR = "lakehouse"
CATALOG_DB = os.path.join(DATA_LAKE_DIR, "hr_lake_catalog.db")
DATA_DIRECTORY = os.path.join(DATA_LAKE_DIR, "hr_lake_data")
SESSION_DB = os.path.join(DATA_LAKE_DIR, "local_session.duckdb")

# Release this notebook's connection before deleting files it may still have open.
if "con" in globals():
    con.close()

if os.path.exists(DATA_LAKE_DIR):
    shutil.rmtree(DATA_LAKE_DIR)
os.makedirs(DATA_DIRECTORY, exist_ok=True)

print(f"Fresh lakehouse created at {os.path.abspath(DATA_LAKE_DIR)}")

Fresh lakehouse created at c:\_cmps360-content\examples\01_de\03_data_lakehouse_basics\lakehouse


In [2]:
# Create one in-memory DuckDB session for query execution.
# DuckLake keeps the catalog and Parquet data persistent on disk.
con = duckdb.connect()
con.execute("INSTALL ducklake")
con.execute("LOAD ducklake")

# Absolute paths avoid surprises when VS Code uses a different working directory.
con.execute(f"""
    ATTACH '{os.path.abspath(CATALOG_DB)}' AS hr_lake (
        TYPE DUCKLAKE,
        DATA_PATH '{os.path.abspath(DATA_DIRECTORY)}'
    )
""")
con.execute("USE hr_lake")

print(f"DuckDB is ready; DuckLake data path: {os.path.abspath(DATA_DIRECTORY)}")

DuckDB is ready; DuckLake data path: c:\_cmps360-content\examples\01_de\03_data_lakehouse_basics\lakehouse\hr_lake_data


In [3]:
# Step 1B: The connection cell above loaded DuckLake and mounted the catalog.
# The storage layer is ready.
print("DuckLake catalog attached as hr_lake.")

DuckLake catalog attached as hr_lake.


In [4]:
# Step 1C: Create the table in DuckLake and commit its first data snapshot.
# The explicit commit makes the write boundary visible in this teaching example.
con.execute("""
    CREATE TABLE employees (
        id INTEGER,
        first_name VARCHAR,
        last_name VARCHAR,
        department VARCHAR,
        salary INTEGER
    )
""")
con.execute("""
    INSERT INTO employees VALUES
    (1, 'Tariq', 'Al-Ali', 'Engineering', 32000),
    (2, 'Fatima', 'Al-Zahra', 'Marketing', 24000)
""")
con.commit()
con.sql("SELECT * FROM employees ORDER BY id").show()

┌───────┬────────────┬───────────┬─────────────┬────────┐
│  id   │ first_name │ last_name │ department  │ salary │
│ int32 │  varchar   │  varchar  │   varchar   │ int32  │
├───────┼────────────┼───────────┼─────────────┼────────┤
│     1 │ Tariq      │ Al-Ali    │ Engineering │  32000 │
│     2 │ Fatima     │ Al-Zahra  │ Marketing   │  24000 │
└───────┴────────────┴───────────┴─────────────┴────────┘



## 🔄 Section 2: Transactions (ACID Multi-Statement Writes)

### Concept Explanation
- **Atomicity**: All SQL statements inside `BEGIN TRANSACTION` commit together or roll back cleanly if an error occurs.
- **Consistency**: Protects the catalog and storage layer against partial updates, ensuring corrupted or incomplete writes never become visible.

### Realistic Demonstration Scenarios
1. **Successful Transaction (`COMMIT`)**: Promotes Tariq Al-Ali and inserts a new employee, Omar Al-Farooq.
2. **Failed Transaction (`ROLLBACK`)**: Simulates a realistic payroll calculation error (division by zero during a batch bonus calculation). The exception triggers `ROLLBACK`, leaving the dataset completely unchanged.

In [5]:
# Step 2A: Group a salary update and an employee insert into one commit.
# Both changes become visible together, which demonstrates atomicity.
con.execute("BEGIN TRANSACTION")
con.execute("UPDATE employees SET salary = 35000 WHERE first_name = 'Tariq'")
con.execute("INSERT INTO employees VALUES (3, 'Omar', 'Al-Farooq', 'Engineering', 28000)")
con.execute("COMMIT")
con.sql("SELECT * FROM employees ORDER BY id").show()

┌───────┬────────────┬───────────┬─────────────┬────────┐
│  id   │ first_name │ last_name │ department  │ salary │
│ int32 │  varchar   │  varchar  │   varchar   │ int32  │
├───────┼────────────┼───────────┼─────────────┼────────┤
│     1 │ Tariq      │ Al-Ali    │ Engineering │  35000 │
│     2 │ Fatima     │ Al-Zahra  │ Marketing   │  24000 │
│     3 │ Omar       │ Al-Farooq │ Engineering │  28000 │
└───────┴────────────┴───────────┴─────────────┴────────┘



In [6]:
# Step 2B: Force an error after a salary update and roll back the whole transaction.
# The update to Tariq must not survive the failed bonus calculation.
try:
    con.execute("BEGIN TRANSACTION")
    con.execute("UPDATE employees SET salary = 38000 WHERE id = 1")
    con.execute("UPDATE employees SET salary = salary + (10000 / 0) WHERE department = 'Engineering'")
    con.execute("COMMIT")
except Exception as error:
    con.execute("ROLLBACK")
    print(f"Expected payroll error; transaction rolled back: {error}")

con.sql("SELECT * FROM employees ORDER BY id").show()

Expected payroll error; transaction rolled back: Conversion Error: Type DOUBLE with value inf can't be cast because the value is out of range for the destination type INT32

LINE 1: UPDATE employees SET salary = salary + (10000 / 0) WHERE department = 'Engineering'
                                             ^
┌───────┬────────────┬───────────┬─────────────┬────────┐
│  id   │ first_name │ last_name │ department  │ salary │
│ int32 │  varchar   │  varchar  │   varchar   │ int32  │
├───────┼────────────┼───────────┼─────────────┼────────┤
│     1 │ Tariq      │ Al-Ali    │ Engineering │  35000 │
│     2 │ Fatima     │ Al-Zahra  │ Marketing   │  24000 │
│     3 │ Omar       │ Al-Farooq │ Engineering │  28000 │
└───────┴────────────┴───────────┴─────────────┴────────┘



## ⏱️ Section 3: Time Travel (Querying Historical Snapshots)

### Concept Explanation
- **Snapshot Ledger**: DuckLake tracks immutable snapshot IDs in `hr_lake_catalog.db` whenever a transaction commits.
- **Time Travel Queries**: Query previous versions of a table using `AT (VERSION => n)` without making file copies or restoring database backups.

### Demonstration Steps
1. Inspect the snapshot log using `ducklake_snapshots('hr_lake')`.
2. Query **VERSION 2** (the initial state containing Tariq & Fatima before Omar was added).
3. Compare with the current version.

In [7]:
# Step 3A: Inspect the commit history recorded by DuckLake.
# Each committed change creates a snapshot that can be queried later.
con.sql("""
    SELECT snapshot_id, snapshot_time, commit_message
    FROM ducklake_snapshots('hr_lake')
""").show()

┌─────────────┬───────────────────────────────┬────────────────┐
│ snapshot_id │         snapshot_time         │ commit_message │
│    int64    │   timestamp with time zone    │    varchar     │
├─────────────┼───────────────────────────────┼────────────────┤
│           0 │ 2026-09-15 23:59:46.484145+03 │ NULL           │
│           1 │ 2026-09-16 00:00:24.974439+03 │ NULL           │
│           2 │ 2026-09-16 00:00:25.021862+03 │ NULL           │
│           3 │ 2026-09-16 00:00:31.439614+03 │ NULL           │
└─────────────┴───────────────────────────────┴────────────────┘



In [8]:
# Step 3B: Read the version before Omar was added, then read the current table.
# The historical query is read-only; it does not restore or modify the table.
con.sql("SELECT * FROM employees AT (VERSION => 2)").show()
con.sql("SELECT * FROM employees ORDER BY id").show()

┌───────┬────────────┬───────────┬─────────────┬────────┐
│  id   │ first_name │ last_name │ department  │ salary │
│ int32 │  varchar   │  varchar  │   varchar   │ int32  │
├───────┼────────────┼───────────┼─────────────┼────────┤
│     1 │ Tariq      │ Al-Ali    │ Engineering │  32000 │
│     2 │ Fatima     │ Al-Zahra  │ Marketing   │  24000 │
└───────┴────────────┴───────────┴─────────────┴────────┘

┌───────┬────────────┬───────────┬─────────────┬────────┐
│  id   │ first_name │ last_name │ department  │ salary │
│ int32 │  varchar   │  varchar  │   varchar   │ int32  │
├───────┼────────────┼───────────┼─────────────┼────────┤
│     1 │ Tariq      │ Al-Ali    │ Engineering │  35000 │
│     2 │ Fatima     │ Al-Zahra  │ Marketing   │  24000 │
│     3 │ Omar       │ Al-Farooq │ Engineering │  28000 │
└───────┴────────────┴───────────┴─────────────┴────────┘



## 🧬 Section 4: Zero-Copy Schema Evolution

### Concept Explanation
- **In-Place Metadata Updates**: Adding a column (`ALTER TABLE ... ADD COLUMN`) updates the catalog schema ledger in `hr_lake_catalog.db` without rewriting existing raw Parquet data files in `hr_lake_data/`.
- **Backward Compatibility**: Existing records automatically display `NULL` / `NaN` for newly added columns until updated.

### Demonstration Steps
1. Add `performance_rating DOUBLE` column.
2. Insert a new record (`Layla Mahmoud`) containing all 6 columns.
3. Inspect the evolved table structure.

In [9]:
# Step 4: Add a column through catalog metadata, then insert a row using the new schema.
# Existing rows remain valid and show NULL until a value is supplied for this column.
con.execute("ALTER TABLE employees ADD COLUMN performance_rating DOUBLE")
con.execute("""
    INSERT INTO employees VALUES
    (4, 'Layla', 'Mahmoud', 'HR', 30000, 4.9)
""")
con.commit()
con.sql("SELECT * FROM employees ORDER BY id").show()

┌───────┬────────────┬───────────┬─────────────┬────────┬────────────────────┐
│  id   │ first_name │ last_name │ department  │ salary │ performance_rating │
│ int32 │  varchar   │  varchar  │   varchar   │ int32  │       double       │
├───────┼────────────┼───────────┼─────────────┼────────┼────────────────────┤
│     1 │ Tariq      │ Al-Ali    │ Engineering │  35000 │               NULL │
│     2 │ Fatima     │ Al-Zahra  │ Marketing   │  24000 │               NULL │
│     3 │ Omar       │ Al-Farooq │ Engineering │  28000 │               NULL │
│     4 │ Layla      │ Mahmoud   │ HR          │  30000 │                4.9 │
└───────┴────────────┴───────────┴─────────────┴────────┴────────────────────┘



## 📁 Section 5: Verification & Local Workspace Structure

Inspect your workspace directory structure after running this notebook to observe the decoupled storage design:

- **`lakehouse/hr_lake_catalog.db`**: The DuckDB relational catalog database managing table schemas, active snapshots, transactions, and file tracking.
- **`lakehouse/hr_lake_data/`**: A folder containing raw, queryable, optimized **Parquet files** generated cleanly via DuckDB writes.

In [10]:
# Step 5: Flush any inline data and inspect the physical files DuckLake produced.
# DuckLake may partition files into nested directories, so search recursively.
con.execute("CALL ducklake_flush_inlined_data('hr_lake')")

parquet_files = sorted(
    os.path.join(root, filename)
    for root, _, filenames in os.walk(DATA_DIRECTORY)
    for filename in filenames
    if filename.lower().endswith(".parquet")
)

if parquet_files:
    print(f"Parquet files written under {os.path.abspath(DATA_DIRECTORY)}:")
    for parquet_file in parquet_files:
        print(f"- {os.path.relpath(parquet_file, DATA_DIRECTORY)}")
else:
    print(f"No Parquet files found under {os.path.abspath(DATA_DIRECTORY)}")
    print("DuckLake's catalog-level file list:")
    con.sql("SELECT * FROM ducklake_list_files('hr_lake', 'employees')").show()

Parquet files written under c:\_cmps360-content\examples\01_de\03_data_lakehouse_basics\lakehouse\hr_lake_data:
- main\employees\ducklake-01a0a6df-c10c-7b22-8423-6d16fb9fc6f7.parquet
- main\employees\ducklake-01a0a6df-c10f-79ba-acc9-aeba8a808984.parquet
- main\employees\ducklake-01a0a6df-c11e-75e6-adf8-e8c34770f0d8-delete.parquet


## 📊 Section 6: DuckDB Engine vs. DuckLake Storage Layer Summary

| Feature / Capability | DuckDB Engine | DuckLake Storage Layer (`hr_lake`) |
| :--- | :--- | :--- |
| **Primary Role** | SQL Parsing, Query Execution, In-Memory Computation | Catalog Ledger, Snapshot History, Parquet Data Layout |
| **Catalog Database** | Default DuckDB system catalog | Dedicated `lakehouse/hr_lake_catalog.db` metadata database |
| **Transactions** | Session-level in-memory ACID | Multi-file ACID commits logged to catalog ledger |
| **Time Travel** | Not available for standard table structures | Instant historical queries via `AT (VERSION => n)` |
| **Schema Evolution** | Session table schema changes | Zero-copy schema evolution without rewriting Parquet files |
| **Storage Layout** | Monolithic local `.duckdb` file | Decoupled metadata catalog (`lakehouse/hr_lake_catalog.db`) & folder storage (`lakehouse/hr_lake_data/`) |

## 🏁 Section 7: Key Takeaways & Conclusion

1. **DuckDB Catalog & Storage Decoupling**: DuckLake separates metadata management (`lakehouse/hr_lake_catalog.db`) from raw data storage (`lakehouse/hr_lake_data/`).
2. **ACID Transactions**: Standard SQL transactions (`BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK`) ensure data integrity and rollback failed calculations cleanly.
3. **Time Travel**: Snapshot logs track every commit, allowing instant time-travel queries via `AT (VERSION => n)`.
4. **Zero-Copy Schema Evolution**: Table schemas evolve in-place with `ALTER TABLE` without rewriting existing Parquet data files.

## 🔌 Section 8: Release the Catalog for DuckDB CLI

Run the release cell first, then close the notebook kernel or leave it disconnected from `hr_lake`.

DuckLake stores the data path in the catalog. On Windows, drive-letter casing and trailing slash normalization can make an explicitly supplied `DATA_PATH` look different even when it points to the same folder. Let the catalog provide its own stored path:

```powershell
duckdb
```

```sql
INSTALL ducklake;
LOAD ducklake;

ATTACH 'lakehouse/hr_lake_catalog.db' AS hr_lake (
    TYPE DUCKLAKE
);

SHOW DATABASES;
USE hr_lake;
SHOW TABLES;
SELECT * FROM employees;
```

In [11]:
# Close the notebook connection so the DuckDB CLI can open the catalog.
con.close()
con = None
print("DuckLake catalog released. You can now connect from the DuckDB CLI.")

DuckLake catalog released. You can now connect from the DuckDB CLI.
